# From LCU to QSVT

- Encode a two-term Hamiltonian with LCU.
- Reuse the block encoding for qubitization and a polynomial transformation with QSVT.
- Extend the construction to eight terms and alias-sampling preparation with configurable QROM fanout.

- The examples use [guppy](https://docs.quantinuum.com/guppy/language_guide/language_guide_index.html). Run from a source checkout with development dependencies installed; the checks use `guppyalgos.tests.helpers`.
- The repository default is little endian.


In [1]:
import numpy as np
import zixy.qubit.pauli as zqp
from guppylang import guppy
from guppylang.std.builtins import array, comptime, dagger
from guppylang.std.quantum import qubit

from guppyalgos.algorithms.block_encoding.lcu import (
    LCU, LCUData, build_single_cntrl_select, build_unary_iteration_select,
)
from guppyalgos.algorithms.block_encoding.qubitization import Qubitization
from guppyalgos.algorithms.state_preparation import multiplexor_prep
from guppyalgos.primitives.subroutines.reflection import Reflection
from guppyalgos.primitives.gate_decompositions.cnx.cnx import cnx
from guppyalgos.tests.helpers import assert_allclose_ignorephase, get_unitary_projected
from guppyalgos.algorithms.block_encoding.qsvt import QSVT


## Projecting out the encoded block

- A block encoding stores a matrix inside a larger unitary. Initialize the preparation qubits at zero and project them back onto zero to extract the system block:

$$
M=(\langle0^a|\otimes I)\,U\,(|0^a\rangle\otimes I).
$$

- Each example passes an explicit projection dictionary to `get_unitary_projected`. Its entries follow the circuit's preparation-register order; QSVT also projects its signal qubit onto zero.
- Projection preserves the block's scale. For normalized $|\psi\rangle$, the post-selection probability is $\|M|\psi\rangle\|^2$.
- Compare the extracted matrix with the expected block, allowing an overall global phase.


In [2]:
from guppylang.std.debug import state_output
from guppylang.std.quantum import discard_array
from guppyalgos.utils import qarray


## 1. LCU: encode two terms

Choose $H=0.6X+0.4Y$, with normalization $\lambda=0.6+0.4=1$.
One preparation qubit selects between X and Y:

$$
P|0\rangle=\sqrt{0.6}|0\rangle+\sqrt{0.4}|1\rangle,
\qquad S=|0\rangle\langle0|\otimes X+|1\rangle\langle1|\otimes Y.
$$

`LCU` applies PREPARE, SELECT, then UNPREPARE. The resulting unitary $B$ contains the Hamiltonian in its zero-ancilla block:

$$
\langle0|B|0\rangle=H/\lambda
=\begin{pmatrix}0&0.6-0.4i\\0.6+0.4i&0\end{pmatrix}.
$$

In [3]:
hamiltonian = zqp.RealTermSum.from_str("(0.6, X0), (0.4, Y0)", 1)
data = LCUData.from_hamiltonian(hamiltonian)
prepare = multiplexor_prep(data.amplitudes)
select = build_single_cntrl_select(data)
A = hamiltonian.to_sparse_matrix().toarray() / data.l1_norm


@guppy
def unprepare(prep_qreg: array[qubit, 1]) -> None:
    with dagger:
        prepare(prep_qreg)


@guppy
def encode(prep_qreg: array[qubit, 1], qreg: array[qubit, 1]) -> None:
    LCU(prepare, select, unprepare).compose(prep_qreg, qreg)


encoded_block = get_unitary_projected(
    encode, 1, {"prep": [False]},
    n_extra_qubits=2,
)
assert_allclose_ignorephase(encoded_block, A, threshold=1e-7)
print("LCU block matches H / normalization.")

LCU block matches H / normalization.


## 2. Qubitization: reuse the LCU

Add a reflection on the preparation qubit to make a quantum walk:

$$
W=RB,\qquad R=I-2|0\rangle\langle0|.
$$

`Qubitization` takes the same LCU and the reflection. Two steps encode

$$
\langle0|W^2|0\rangle=T_2(A)=2A^2-I=0.04I,
\qquad A=H/\lambda.
$$

Use `.compose(...)` for one step or `.power(..., d)` for $d$ steps. Keep the preparation register coherent between them.

In [4]:
@guppy
def walk_squared(prep_qreg: array[qubit, 1], qreg: array[qubit, 1]) -> None:
    Qubitization(
        LCU(prepare, select, unprepare), Reflection[1, 0](cnx)
    ).power(prep_qreg, qreg, 2)


walk_block = get_unitary_projected(
    walk_squared, 1, {"prep": [False]},
    n_extra_qubits=2,
)
assert_allclose_ignorephase(walk_block, 2 * A @ A - np.eye(2), threshold=1e-7)
print("Two walk steps encode 2A² - I.")

Two walk steps encode 2A² - I.


## 3. QSVT: choose a polynomial

`QSVT` combines the LCU, its adjoint and a phase sequence. Here $B^\dagger=B$, so both arguments use the same LCU. The two half-turn phases `[1.0, 1.0]` give

$$
p(x)=1-2x^2,\qquad
\langle0|_s\langle0|_p V|0\rangle_s|0\rangle_p=p(A)=-0.04I.
$$

The extra signal qubit and the preparation qubit are both projected onto zero to expose this block. This particular Hamiltonian has $A^2=0.52I$, which makes the result a multiple of the identity.

The diagnostic circuits below allocate two scratch qubits after QSVT to reset discarded measurement workspace for statevector extraction. This cleanup is separate from the preparation and signal projections.

In [5]:
phases = [1.0, 1.0]


@guppy
def transform(
    prep_qreg: array[qubit, 1], signal_qreg: array[qubit, 1], qreg: array[qubit, 1],
) -> None:
    QSVT(
        LCU(prepare, select, unprepare),
        LCU(prepare, select, unprepare),
        comptime(phases),
    ).compose(signal_qreg[0], prep_qreg, qreg)
    # Reset discarded measurement workspace before extracting a statevector block.
    scratch_qreg = qarray(2)
    state_output("cleared_workspace", scratch_qreg)
    discard_array(scratch_qreg)


qsvt_block = get_unitary_projected(
    transform, 1, {"prep": [False], "signal": [False]},
    n_extra_qubits=2,
)
assert_allclose_ignorephase(qsvt_block, np.eye(2) - 2 * A @ A, threshold=1e-7)
print("QSVT block matches I - 2A².")

QSVT block matches I - 2A².


## 4. Replace the Hamiltonian

For an arbitrary Hermitian Pauli sum $H=\sum_j a_jP_j$, use `LCUData` to obtain $\lambda=\sum_j|a_j|$ and the register sizes. `build_unary_iteration_select` handles the term selection. The QSVT composition stays the same.

For example, take

$$
\begin{aligned}
H={}&0.25Z_0-0.125X_1+0.125Y_0Y_1+0.125Z_0X_1\\
&+0.125X_0+0.125Z_1+0.0625X_0Z_1-0.0625Z_0Z_1.
\end{aligned}
$$

Eight terms need three preparation qubits and two system qubits. The same phases still encode $p(H/\lambda)=I-2(H/\lambda)^2$, now a nontrivial four-by-four matrix.

In [6]:
larger_hamiltonian = zqp.RealTermSum.from_str(
    "(0.25, Z0), (-0.125, X1), (0.125, Y0 Y1), (0.125, Z0 X1), "
    "(0.125, X0), (0.125, Z1), (0.0625, X0 Z1), (-0.0625, Z0 Z1)", 2
)
larger_data = LCUData.from_hamiltonian(larger_hamiltonian)
n_prep = larger_data.n_prep_qubits
n_state = larger_data.n_state_qubits
larger_prepare = multiplexor_prep(larger_data.amplitudes)
larger_select = build_unary_iteration_select(larger_data)


@guppy
def larger_unprepare(prep_qreg: array[qubit, n_prep]) -> None:
    with dagger:
        larger_prepare(prep_qreg)


@guppy
def larger_transform(
    prep_qreg: array[qubit, n_prep], signal_qreg: array[qubit, 1],
    qreg: array[qubit, n_state],
) -> None:
    QSVT(
        LCU(larger_prepare, larger_select, larger_unprepare),
        LCU(larger_prepare, larger_select, larger_unprepare),
        comptime(phases),
    ).compose(signal_qreg[0], prep_qreg, qreg)
    # Reset discarded measurement workspace before extracting a statevector block.
    scratch_qreg = qarray(2)
    state_output("cleared_workspace", scratch_qreg)
    discard_array(scratch_qreg)


# Use the same matrix ordering as get_unitary_projected.
larger_A = larger_hamiltonian.to_sparse_matrix(big_endian=True).toarray() / larger_data.l1_norm
larger_block = get_unitary_projected(
    larger_transform, n_state,
    {"prep": [False] * n_prep, "signal": [False]},
    n_extra_qubits=2,
)
assert_allclose_ignorephase(
    larger_block, np.eye(2**n_state) - 2 * larger_A @ larger_A,
    threshold=1e-7,
)
print("Eight-term QSVT block matches I - 2(H / normalization)².")


Eight-term QSVT block matches I - 2(H / normalization)².


## 5. LCU with alias-sampling PREPARE

For a larger table of terms, alias sampling is another way to prepare their weights. Unlike the earlier rotation-based preparation, it retains workspace alongside the index:

$$
P|0\rangle=\sum_j\sqrt{\widetilde p_j}|j\rangle|g_j\rangle,
\qquad p_j=|a_j|/\lambda.
$$

SELECT acts on the index and system; UNPREPARE reverses alias sampling on **all** preparation registers. Together they give

$$
\langle0|P^\dagger SP|0\rangle
=\sum_j\widetilde p_j\operatorname{sgn}(a_j)P_j.
$$

We reuse the eight-term Hamiltonian above. Its weights are exactly representable with four probability bits, so $\widetilde p_j=p_j$ and the block is $H$ ($\lambda=1$). Other weights are rounded at the chosen precision.

In [7]:
from guppyalgos.algorithms.state_preparation.alias_sampling import (
    AliasSamplingRegs, alias_samp_prep,
)
from guppyalgos.primitives.subroutines.fanout import (
    fanout_basic, fanout_measurement_parity,
)

alias_hamiltonian = larger_hamiltonian
alias_data = LCUData.from_hamiltonian(alias_hamiltonian)
alias_probabilities = np.abs(alias_data.coeffs) / alias_data.l1_norm
alias_select = build_unary_iteration_select(alias_data)


def build_alias_lcu(fanout_op=fanout_basic):
    alias_prepare = alias_samp_prep(
        alias_probabilities, precision=1 / 16, fanout_op=fanout_op,
    )

    @guppy
    def prepare_alias(regs: AliasSamplingRegs[3, 4]) -> None:
        alias_prepare(
            regs.index, regs.alternative, regs.keep,
            regs.comparison, regs.comparison_result, False,
        )

    @guppy
    def select_alias(regs: AliasSamplingRegs[3, 4], qreg: array[qubit, 2]) -> None:
        alias_select(regs.index, qreg)

    @guppy
    def unprepare_alias(regs: AliasSamplingRegs[3, 4]) -> None:
        alias_prepare(
            regs.index, regs.alternative, regs.keep,
            regs.comparison, regs.comparison_result, True,
        )

    @guppy
    def encode_alias(regs: AliasSamplingRegs[3, 4], qreg: array[qubit, 2]) -> None:
        LCU(prepare_alias, select_alias, unprepare_alias).compose(regs, qreg)

    return encode_alias


alias_encode = build_alias_lcu()

alias_encode.compile_function()
print("Alias LCU with persistent preparation workspace compiled.")


Alias LCU with persistent preparation workspace compiled.


`AliasSamplingRegs[3, 4]` contains a three-qubit index, a three-qubit alternative index, two four-qubit probability registers and one comparison flag: **15 preparation qubits** in total. Initialize all of them to zero and retain them through SELECT and UNPREPARE. The encoded block projects all 15 onto zero; the workspace must not be discarded between these steps.

UNPREPARE alone does not guarantee clean workspace after SELECT. For this Hamiltonian acting on the system state $|00\rangle$, the comparison register is nonzero with probability $3/16$ after the full LCU. Keep it through every subsequent coherent algorithm step; discarding it would remove coherence.


## 6. Change the alias QROM fanout

Alias PREPARE uses QROM to load an alternative index and a keep threshold for each address. Its lookup XORs these words into two workspace registers:

$$
|j\rangle|u\rangle|v\rangle
\longmapsto
|j\rangle|u\oplus\mathrm{alt}_j\rangle|v\oplus\mathrm{keep}_j\rangle.
$$

`alias_samp_prep(..., fanout_op=...)` chooses how the active address flag controls the stored one-bits. Its `build_alias_fanout` adapter gathers selected bits from both registers into one fanout call.

- `fanout_basic`: sequential CNOTs.
- `fanout_measurement_parity`: measurement-assisted fanout with feed-forward; four or more selected bits use extra ancillas.

Both implement the same lookup on arbitrary workspace states. This lets us change the QROM implementation without changing the probabilities, SELECT or LCU interface. UNPREPARE uses the alias routine's explicit inverse mode, including the same QROM choice.

In [8]:
# Choose the fanout when constructing alias PREPARE.
measurement_prepare = alias_samp_prep(
    alias_probabilities,
    precision=1 / 16,
    fanout_op=fanout_measurement_parity,
)

# The LCU factory forwards the same option to PREPARE and UNPREPARE.
measurement_alias_encode = build_alias_lcu(
    fanout_op=fanout_measurement_parity,
)
measurement_alias_encode.compile_function()
print("Alias LCU with measurement-assisted QROM compiled.")

Alias LCU with measurement-assisted QROM compiled.


`fanout_log` is another option when every alias-table row selects at least one bit. The current implementation does not support empty fanouts, so it is not a drop-in choice for arbitrary alias tables. Use the basic or parity implementation when rows may be all zero.